In [ ]:
import os, random, pickle, itertools, zipfile
from datetime import datetime
import numpy as np

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ================== 1) PyTorch & TorchVision ==================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from typing import Dict, List
from PIL import Image

print(f"[INFO] Started at {datetime.now().isoformat()}")

# ================== 2) CUDA & Seeds  ==================
print("[INFO] torch.cuda.is_available() ->", torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _soft_cuda_sanity():
    try:
        if device.type == 'cuda':
            _x = torch.randn(4, device='cuda').sum().item()
            _name = torch.cuda.get_device_name(0)
            _cap  = torch.cuda.get_device_capability(0)
            _mem  = torch.cuda.memory_reserved() / (1024**2)
            print(f"[INFO] CUDA sanity OK on device: {_name} | cap={_cap} | reserved={_mem:.1f} MB")
            return True, ""
        else:
            return False, "CUDA not available"
    except Exception as e:
        return False, repr(e)

ok, err = _soft_cuda_sanity()
if not ok and device.type == 'cuda':
    print("[WARN] CUDA sanity failed:", err)
    print("[WARN] Falling back to CPU. Tip: If you saw 'device-side assert triggered', do Runtime→Restart runtime.")
    device = torch.device('cpu')

print("[INFO] Using device:", device)
if device.type == 'cuda':
    print("[INFO] CUDA device:", torch.cuda.get_device_name(0))
    print("[INFO] CUDA capability:", torch.cuda.get_device_capability(0))
    print("[INFO] CUDA current mem (MB):", torch.cuda.memory_reserved() / (1024**2))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == 'cuda': torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
print(f"[INFO] Seeds set to {SEED}")

# ================== 3) paths ==================
# model_path = "/content/drive/.../expriment_GN_task1_best_test_for_finetune_tiny_imageNet.pth"
model_path      = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task1_best_test_for_finetune_tiny_imageNet_temp.pth"
topk_path       = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task1F_tiny_imageNet_topk_temp.pkl"
neighbors_path  = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task1F_tiny_imageNet_neighbors_temp.pkl"
ckpt_dir        = "/content/drive/MyDrive/ML_Project/project_files/Group_norm"
os.makedirs(ckpt_dir, exist_ok=True)

def _check_file(path, tag):
    if not os.path.exists(path): raise FileNotFoundError(f"[MISSING] {tag}: {path}")
    print(f"[OK] {tag} exists ({os.path.getsize(path)/(1024**2):.2f} MB): {path}")

_check_file(model_path, "Checkpoint(task1)")
_check_file(topk_path, "Fisher Top-K")
_check_file(neighbors_path, "Fisher Neighbors")

# ================== 4) Tiny-ImageNet (processed .npy) ==================
ZIP_PATH  = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/tiny-imagenet-processed.zip"
DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")
    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP not found at:\n{zip_path}")
    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"
    return data_root

ensure_extracted(ZIP_PATH, DATA_ROOT)

TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):
    """
    expects:
      processed/x_train_01.npy ... x_train_20.npy
      processed/y_train_01.npy ... y_train_20.npy
      processed/x_val_01.npy   ... x_val_20.npy
      processed/y_val_01.npy   ... y_val_20.npy
    """
    def __init__(self, root: str, train: bool=True, transform=None):
        self.root = root; self.train = train; self.transform = transform
        split = "train" if self.train else "val"
        xs, ys = [], []
        for num in range(20):
            xs.append(np.load(os.path.join(root, f'processed/x_{split}_{num+1:02d}.npy')))
            ys.append(np.load(os.path.join(root, f'processed/y_{split}_{num+1:02d}.npy')))
        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])

        if img.ndim == 3 and img.shape[0] == 3 and (img.shape[-1] != 3):
            img = np.transpose(img, (1, 2, 0))

        if img.ndim == 2:
            img = np.stack([img, img, img], axis=-1)
        elif img.ndim == 3 and img.shape[-1] == 1:
            img = np.repeat(img, 3, axis=-1)

        if img.dtype != np.uint8:
            vmin, vmax = float(img.min()), float(img.max())
            if vmax <= 1.5 and vmin >= -0.1:
                img = (np.clip(img, 0, 1) * 255.0).astype(np.uint8)
            else:
                img = np.clip(img, 0, 255).astype(np.uint8)

        img = Image.fromarray(img)
        if self.transform is not None:
            img = self.transform(img)
        return img, target

tf_train = transforms.Compose([
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
])
tf_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
])

print("[INFO] Loading Tiny-ImageNet ...")
train_set = TinyImagenet(root=DATA_ROOT, train=True,  transform=tf_train)
test_set  = TinyImagenet(root=DATA_ROOT, train=False, transform=tf_test)
print(f"[INFO] Train size={len(train_set)} | Val/Test size={len(test_set)}")

def build_tasks(num_classes: int = 200, classes_per_task: int = 20) -> Dict[str, List[int]]:
    assert num_classes % classes_per_task == 0
    n_tasks = num_classes // classes_per_task  # 10
    tasks = {}
    for t in range(n_tasks):
        start = t * classes_per_task
        tasks[f"task{t+1}"] = list(range(start, start + classes_per_task))
    return tasks

# ===== tasks =====
tasks = build_tasks(num_classes=200, classes_per_task=20)
task1_classes = tasks["task1"]          # [0..19]
task2_classes = tasks["task2"]          # [20..39]

# ================== Dataset wrapper for label remapping to 0..K-1 ==================
class RemapView(Dataset):
    """
    Wraps a base dataset and a set of indices, then remaps labels
    according to class_to_new.
    """
    def __init__(self, base_ds: Dataset, indices: List[int], class_to_new: Dict[int, int]):
        self.base = base_ds
        self.indices = list(indices)
        self.class_to_new = {int(k): int(v) for k, v in class_to_new.items()}

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        x, y_orig = self.base[self.indices[i]]
        y_new = self.class_to_new[int(y_orig)]
        return x, y_new

def build_remapped_subset(dataset, keep_classes: List[int]) -> RemapView:
    keep_set = set(int(c) for c in keep_classes)
    idx = [i for i, y in enumerate(dataset.targets) if int(y) in keep_set]
    class_to_new = {c: i for i, c in enumerate(sorted(keep_set))}
    print(f"[DEBUG][RemapView] kept={len(idx)} samples | classes={sorted(keep_set)} | map_head={list(class_to_new.items())[:5]}")
    return RemapView(dataset, idx, class_to_new)

train_23_full = build_remapped_subset(train_set, task2_classes)
test_23       = build_remapped_subset(test_set,  task2_classes)
test_01       = build_remapped_subset(test_set,  task1_classes)

def make_loader(ds, bs, shuffle, seed=SEED, num_workers=2):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g,
                      num_workers=num_workers, pin_memory=(device.type=='cuda'),
                      persistent_workers=(num_workers>0))



# ================== 5)  (ResNet18 + GroupNorm) ==================
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 3, stride, 1, bias=False)

def _gn(num_channels: int, num_groups: int = 32):
    return nn.GroupNorm(num_groups, num_channels)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1   = _gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2   = _gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                _gn(planes)
            )
    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        return torch.relu(out)

class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()
        self.nf = nf
        self.conv1 = conv3x3(3, nf)
        self.gn1   = _gn(nf)
        self.layer1 = nn.Sequential(BasicBlock(nf, nf, 1),       BasicBlock(nf, nf, 1))
        self.layer2 = nn.Sequential(BasicBlock(nf, nf*2, 2),     BasicBlock(nf*2, nf*2, 1))
        self.layer3 = nn.Sequential(BasicBlock(nf*2, nf*4, 2),   BasicBlock(nf*4, nf*4, 1))
        self.layer4 = nn.Sequential(BasicBlock(nf*4, nf*8, 2),   BasicBlock(nf*8, nf*8, 1))
    def forward(self, x):
        x = torch.relu(self.gn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = torch.nn.functional.avg_pool2d(x, x.shape[2]); x = x.view(x.size(0), -1)
        return x
    @property
    def out_dim(self): return self.nf*8

class MultiHeadNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()
    def add_head(self, name, num_classes):
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[name] = head
    def forward(self, x, head):
        feat = self.backbone(x)
        return self.heads[head](feat)

# ================== 6) Top-K & Neighbors ==================
with open(topk_path, "rb") as f: topk_fisher = pickle.load(f)
with open(neighbors_path, "rb") as f: fisher_neighbors = pickle.load(f)
TOPK_COUNT = len(topk_fisher)
NEIGH_COUNT = len(fisher_neighbors)

# ================== 7) EWC & Freeze Helpers ==================
def build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values):
    pm = dict(model.named_parameters())
    usable = [n for n in fisher_neighbors if n['name'] in pm]
    if not usable: return {}
    max_f = max((n['fisher'] for n in usable), default=1.0) or 1.0
    buckets = {}
    for n in usable:
        buckets.setdefault(n['name'], []).append((int(n['index']), float(n['fisher'])/max_f))
    ewc = {}
    for name, lst in buckets.items():
        lst.sort(key=lambda t:t[0])
        idxs = torch.tensor([i for i,_ in lst], device=device, dtype=torch.long)
        fish = torch.tensor([f for _,f in lst], device=device, dtype=torch.float32)
        flat = pm[name].view(-1)
        orig = torch.stack([neighbor_original_values[(name, int(i))] for i in idxs.tolist()]).to(flat.device, dtype=flat.dtype)
        ewc[name] = {'idxs': idxs, 'fish': fish, 'orig': orig}
    return ewc

def build_freeze_masks_and_cache(model, topk_list):
        """
    Freeze all Top-K parameters except:
      - The current task head: heads.task2.*

    GroupNorm has no running statistics, but its weights and biases
    are handled like regular trainable parameters.
    """
    masks, frozen_idxs, frozen_vals = {}, {}, {}
    pm = dict(model.named_parameters())

    by_name = {}
    for e in topk_list:
        n, i = e['name'], int(e['index'])
        if (n in pm) and (not n.startswith("heads.task2.")):
            by_name.setdefault(n, []).append(i)

    for name, idxs in by_name.items():
        p = pm[name]
        flat = p.detach().view(-1)
        idxs_t = torch.tensor(idxs, device=flat.device, dtype=torch.long)

        if p.requires_grad:
            m = torch.ones_like(p, dtype=torch.bool, device=p.device)
            mv = m.view(-1); mv[idxs_t] = False
            masks[name] = mv.view_as(m)

        with torch.no_grad():
            frozen_idxs[name] = idxs_t
            frozen_vals[name] = flat.index_select(0, idxs_t).clone()

    return masks, frozen_idxs, frozen_vals

def apply_freeze_after_backward(model, masks):
    with torch.no_grad():
        for n,p in model.named_parameters():
            m = masks.get(n, None)
            if p.grad is not None and m is not None:
                p.grad.mul_(m.to(p.grad.dtype))

@torch.no_grad()
def apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals):
    for n, idxs in frozen_idxs.items():
        if n in param_map:
            flat = param_map[n].view(-1)
            flat.index_copy_(0, idxs, frozen_vals[n].to(flat.device, dtype=flat.dtype))

def mask_stats(masks):
    total = sum(m.numel() for m in masks.values())
    frozen = sum((~m).sum().item() for m in masks.values())
    return total, frozen

def audit_topk_vs_masks(model, topk_list):
    pm = dict(model.named_parameters())
    unique_pairs = set((e['name'], int(e['index'])) for e in topk_list)
    excl_not_found = excl_head_t2 = excl_no_grad = excl_oob = 0
    included = set()
    for name, idx in unique_pairs:
        p = pm.get(name, None)
        if p is None:
            excl_not_found += 1; continue
        if name.startswith("heads.task2."):
            excl_head_t2 += 1; continue
        if idx < 0 or idx >= p.numel():
            excl_oob += 1; continue
        if not p.requires_grad:
            excl_no_grad += 1
        included.add((name, idx))
    print(f"[AUDIT] TopK unique pairs     : {len(unique_pairs)}")
    print(f"[AUDIT] Excluded heads.task2.*: {excl_head_t2}")
    print(f"[AUDIT] Not found             : {excl_not_found}")
    print(f"[AUDIT] Out-of-bounds         : {excl_oob}")
    print(f"[AUDIT] No-grad params        : {excl_no_grad}")
    print(f"[AUDIT] Will be masked/strict : {len(included)}")

def freeze_backbone_bn_running_stats(model):
        # GroupNorm has no running statistics, so this function has no effect.
    # It is kept unchanged for easier comparison with the previous BN-based version.
    return

# ================== 8) Initialize model and regularizers ==================
def init_model_and_regularizers(lambda_ewc):
    backbone = ResNet18Backbone(nf=64).to(device)
    model = MultiHeadNet(backbone).to(device)
    for t in range(1, 10+1):
        model.add_head(f"task{t}", 20)
    model.to(device)

    ckpt = torch.load(model_path, map_location=device)
    sd = ckpt["model_state"] if "model_state" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:    print("[INIT] Missing keys:", missing)
    if unexpected: print("[INIT] Unexpected keys:", unexpected)

        # (1) Freeze the Task 1 head while training Task 2
    for p in model.heads["task1"].parameters():
        p.requires_grad = False

        # (2) Warm-start the Task 2 head from Task 1
    with torch.no_grad():
        if ("task1" in model.heads) and ("task2" in model.heads):
            h1 = model.heads["task1"]; h2 = model.heads["task2"]
            same_W = (h1.weight.shape == h2.weight.shape)
            same_b = (h1.bias is not None) and (h2.bias is not None) and (h1.bias.shape == h2.bias.shape)
            if same_W: h2.weight.copy_(h1.weight)
            if same_b: h2.bias.copy_(h1.bias)
            print(f"[INIT] Copied task1 → task2 head | W={same_W} | b={same_b}")
        else:
            print("[INIT] WARN: task1/task2 head not found — skip head weight copy")

        # (3) Prepare EWC reference values
    with torch.no_grad():
        cpu_cache = {n: p.view(-1).detach().cpu() for n,p in model.named_parameters()}

    neighbor_original_values = {}
    for n in fisher_neighbors:
        name, idx = n['name'], int(n['index'])
        if name in cpu_cache and idx < cpu_cache[name].numel():
            neighbor_original_values[(name, idx)] = cpu_cache[name][idx]

    ewc_tensors = build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values)

        # (4) Build Top-K freeze masks
    masks, frozen_idxs, frozen_vals = build_freeze_masks_and_cache(model, topk_fisher)

    audit_topk_vs_masks(model, topk_fisher)
    return model, ewc_tensors, masks, frozen_idxs, frozen_vals

# ================== 9) Evaluation ==================
@torch.no_grad()
def evaluate_head(model, loader, head):
    model.eval(); correct=0; total=0
    for x,y in loader:
        x = x.to(device); y = torch.as_tensor(y, device=device, dtype=torch.long)
        logits = model(x, head=head); pred = logits.argmax(1)
        correct += (pred==y).sum().item(); total += y.size(0)
    return 100.0*correct/max(1,total)

# ================== 10) Train one setting ==================
def train_one_setting(lr_backbone, lr_head, bs, lambda_ewc, epochs):
    model, ewc_tensors, masks, frozen_idxs, frozen_vals = init_model_and_regularizers(lambda_ewc)

    # Split optimizer parameters into four groups
    bb_decay, bb_nodecay, hd_decay, hd_nodecay = [], [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_head2 = n.startswith("heads.task2.")
        no_decay = (p.dim()==1) or n.endswith(".bias") or ("gn" in n.lower()) or ("bn" in n.lower())
        if is_head2:
            (hd_nodecay if no_decay else hd_decay).append(p)
        else:
            (bb_nodecay if no_decay else bb_decay).append(p)

    optimizer = optim.SGD(
        [
            {"params": bb_decay,   "lr": lr_backbone, "weight_decay": 5e-4},
            {"params": bb_nodecay, "lr": lr_backbone, "weight_decay": 0.0},
            {"params": hd_decay,   "lr": lr_head,     "weight_decay": 0.0},
            {"params": hd_nodecay, "lr": lr_head,     "weight_decay": 0.0},
        ],
        momentum=0.9
    )

    param_map = {n:p for n,p in model.named_parameters()}

    train_loader = make_loader(train_23_full, bs=bs, shuffle=True)
    test1_loader = make_loader(test_01,       bs=256, shuffle=False)
    test2_loader = make_loader(test_23,       bs=256, shuffle=False)

# ================== [DEBUG] Verify label remapping and head outputs ==================
    try:
        out_feats = model.heads["task2"].out_features
    except Exception as e:
        raise RuntimeError("[DEBUG] head 'task2' not found") from e
    print(f"[DEBUG] head task2 out_features = {out_feats}")
    assert out_feats == 20, f"[DEBUG] task2 head must have 20 classes, got {out_feats}"

    check_labels_range(train_23_full, 0, 19, "train_task2")
    check_labels_range(test_23,       0, 19, "test_task2")
    check_labels_range(test_01,       0, 19, "test_task1")

    from torch.utils.data import DataLoader as _DL
    debug_first_batch(model, _DL(train_23_full, batch_size=min(512, bs), shuffle=True, num_workers=2),
                      head_name="task2", num_classes=20)

    pre_t1 = evaluate_head(model, test1_loader, head="task1")
    print(f"[PRE] Task1 accuracy BEFORE fine-tuning Task2: {pre_t1:.2f}%")

    total_mask_elems, total_frozen = mask_stats(masks)
    print(f"[INFO]   mask_elems={total_mask_elems} | frozen(TopK)={total_frozen}")
    print(f"[OPT ]   lr_backbone={lr_backbone} | lr_head={lr_head} | "
          f"bb_decay={len(bb_decay)} bb_nodecay={len(bb_nodecay)} | "
          f"hd_decay={len(hd_decay)} hd_nodecay={len(hd_nodecay)}")

    best = {'epoch': -1, 't1': -1.0, 't2': -1.0, 'avg': -1.0, 'model_state': None}
    for e in range(1, epochs+1):
        model.train()
        freeze_backbone_bn_running_stats(model)

        running_loss=0.0; steps=0
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = torch.as_tensor(labels, device=device, dtype=torch.long)
            optimizer.zero_grad(set_to_none=True)

            # ================== [DEBUG] Safety check for batch labels ==================
            if (labels.min() < 0) or (labels.max() >= 20):
                bad = labels[(labels < 0) | (labels >= 20)]
                print("[DEBUG][BATCH] Out-of-range labels detected!"
                      f" min={int(labels.min())} max={int(labels.max())} | sample={bad[:20].tolist()}")
                raise RuntimeError("[DEBUG] Task2 expects labels in 0..19. Check the subset/remapping.")

            # EWC penalty
            ewc_penalty = 0.0
            if lambda_ewc != 0 and len(ewc_tensors) > 0:
                for name, pack in ewc_tensors.items():
                    p = param_map[name].view(-1)
                    diff = p.index_select(0, pack['idxs']) - pack['orig']
                    ewc_penalty += (pack['fish'] * (diff**2)).sum()

            logits = model(imgs, head="task2")

           # ================== [DEBUG] Verify logits shape ==================
            if logits.shape[1] != 20:
                raise RuntimeError(f"[DEBUG] logits.shape[1]={logits.shape[1]} != 20 (head/task mismatch)")

            loss = nn.functional.cross_entropy(logits, labels) + (lambda_ewc/2.0)*ewc_penalty

            loss.backward()
            apply_freeze_after_backward(model, masks)
            optimizer.step()
            apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals)

            running_loss += float(loss.detach().cpu()); steps += 1

        acc_t1 = evaluate_head(model, test1_loader, head="task1")
        acc_t2 = evaluate_head(model, test2_loader, head="task2")
        avg = 0.5*(acc_t1 + acc_t2)
        print(f"  Epoch {e}/{epochs} | train_loss={running_loss/max(1,steps):.4f} | "
              f"T1={acc_t1:.2f}% T2={acc_t2:.2f}% AVG={avg:.2f}%")

        # ================== [DEBUG] Quick performance monitoring ==================
        if avg > best['avg'] or (avg == best['avg'] and acc_t2 > best['t2']):
            best = {'epoch': e, 't1': acc_t1, 't2': acc_t2, 'avg': avg,
                    'model_state': {k:v.detach().cpu() for k,v in model.state_dict().items()}}

    return best

# ================== 11) GRID SEARCH  ==================
def run_grid_search_and_save_best():
    backbone_lrs = [0.00001]
    head_lrs     = [0.7]
    batch_sizes  = [32]
    epoch_counts   = [250]
    lambda_values  = [2]

    print(f"[DATA ] Train(T2)={len(train_23_full)} | Test(T2)={len(test_23)} | Test(T1)={len(test_01)}")
    print(f"[META ] TopK={TOPK_COUNT} | Neighbors={NEIGH_COUNT}")

    global_best = {'avg': -1.0, 't1': -1.0, 't2': -1.0, 'epoch': -1, 'setting': None, 'state': None}

    for lr_bb, lr_hd, bs, lam, ep in itertools.product(backbone_lrs, head_lrs, batch_sizes, lambda_values, epoch_counts):
        print(f"\n[SETTING] LR_backbone={lr_bb}, LR_head={lr_hd}, BS={bs}, λ={lam}, EPOCHS={ep} "
              f"| TopK={TOPK_COUNT}, Neigh={NEIGH_COUNT}")
        best = train_one_setting(lr_backbone=lr_bb, lr_head=lr_hd, bs=bs, lambda_ewc=lam, epochs=ep)
        print(f"[SETTING-BEST] epoch {best['epoch']} | T1={best['t1']:.2f}% | T2={best['t2']:.2f}% | AVG={best['avg']:.2f}%")
        if best['avg'] > global_best['avg']:
            global_best = {
                'avg': best['avg'], 't1': best['t1'], 't2': best['t2'],
                'epoch': best['epoch'],
                'setting': f"LR_backbone={lr_bb}, LR_head={lr_hd}, BS={bs}, λ={lam}, EPOCHS={ep}",
                'state': best['model_state']
            }

    best_test_path = os.path.join(ckpt_dir, "Gtask2_best_test_for_finetune_ResNet18_GN_Temp2.pth")
    torch.save({
        "model_state": global_best['state'],      # BEST-TEST weights (state_dict)
        "best_config": global_best['setting'],
        "best_test_acc": global_best['t2'],
        "best_test_epoch": global_best['epoch'],
        "trained_head": "task2",
        "all_heads": [f"task{i}" for i in range(1, 11)],
    }, best_test_path)

    print("\n====================")
    print(f"[GLOBAL BEST] AVG={global_best['avg']:.2f}% | T1={global_best['t1']:.2f}% | T2={global_best['t2']:.2f}% | at epoch {global_best['epoch']}")
    print(f"[SETTING     ] {global_best['setting']}")
    print(f"[SAVED       ] {best_test_path}")

# ================== 12) run ==================
run_grid_search_and_save_best()


Mounted at /content/drive
[INFO] Started at 2026-09-04T13:16:31.303417
[INFO] torch.cuda.is_available() -> True
[INFO] CUDA sanity OK on device: NVIDIA A100-SXM4-40GB | cap=(8, 0) | reserved=1874.0 MB
[INFO] Using device: cuda
[INFO] CUDA device: NVIDIA A100-SXM4-40GB
[INFO] CUDA capability: (8, 0)
[INFO] CUDA current mem (MB): 1874.0
[INFO] Seeds set to 42
[OK] Checkpoint(task1) exists (43.03 MB): /content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task1_best_test_for_finetune_tiny_imageNet_temp.pth
[OK] Fisher Top-K exists (26.37 MB): /content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task1F_tiny_imageNet_topk_temp.pkl
[OK] Fisher Neighbors exists (23.70 MB): /content/drive/MyDrive/ML_Project/project_files/Group_norm/expriment2_GN_task1F_tiny_imageNet_neighbors_temp.pkl
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
[INFO] Loading Tiny-ImageNet ...
[INFO] Train size=100000 | Val/Test size=10000
[DEBU